In [6]:
# crawl_laodong_kinhdoanh_from71.py
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import csv
import os
import time
import random
import re

START_PAGE = 1      # trang bắt đầu
MAX_PAGE = 5000       # giới hạn trên để tránh loop vô hạn (thay đổi nếu cần)
TARGET = 1500         # số bài unique mong muốn
BASE_URL = "https://laodong.vn/kinh-doanh?page={}"
OUTPUT_CSV = "laodong_congnghe.csv"

def make_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-extensions")
    options.add_argument("--disable-notifications")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument(
        "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(options=options)

def extract_article(driver, url):
    try:
        driver.get(url)
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        time.sleep(0.5 + random.random() * 0.7)
    except Exception as e:
        print("   ⚠️ Không load được:", url, "|", e)
        return None

    # Title
    title = ""
    try:
        # cố gắng lấy title phổ biến
        title = driver.find_element(By.CSS_SELECTOR, "h1").text.strip()
    except:
        try:
            title = driver.find_element(By.CSS_SELECTOR, "div.wrapper-01").text.strip()
        except:
            title = ""

    # Body paragraphs
    paragraphs = []
    try:
        body_div = driver.find_element(By.CSS_SELECTOR, "div#gallery-ctt.art-body")
        p_tags = body_div.find_elements(By.TAG_NAME, "p")
        for p in p_tags:
            t = p.text.strip()
            if t:
                paragraphs.append(t)
    except:
        # fallback to generic .art-body
        try:
            body_div = driver.find_element(By.CSS_SELECTOR, "div.art-body")
            p_tags = body_div.find_elements(By.TAG_NAME, "p")
            for p in p_tags:
                t = p.text.strip()
                if t:
                    paragraphs.append(t)
        except:
            pass

    body = "\n\n".join(paragraphs).strip()
    context = (title + "\n\n" + body).strip() if (title or body) else ""

    return {"url": url, "title": title, "context": context}

def load_seen_urls_from_csv(path):
    seen = set()
    if not os.path.exists(path):
        return seen
    try:
        with open(path, "r", encoding="utf-8-sig", newline="") as f:
            reader = csv.DictReader(f)
            for r in reader:
                u = r.get("url")
                if u:
                    seen.add(u.strip())
    except Exception as e:
        print("⚠️ Lỗi khi đọc file CSV hiện có:", e)
    return seen

def crawl():
    driver = make_driver()

    # load các url đã có (nếu file tồn)
    seen_urls = load_seen_urls_from_csv(OUTPUT_CSV)
    current_count = len(seen_urls)
    print(f"🔸 Đã có sẵn {current_count} bài trong {OUTPUT_CSV}")

    # mở file để append (nếu chưa có header thì thêm)
    file_exists = os.path.exists(OUTPUT_CSV)
    csv_file = open(OUTPUT_CSV, "a", newline="", encoding="utf-8-sig")
    writer = csv.DictWriter(csv_file, fieldnames=["url", "title", "context"])
    if not file_exists:
        writer.writeheader()
        csv_file.flush()

    total_added = 0
    page = START_PAGE

    try:
        while current_count < TARGET and page <= MAX_PAGE:
            page_url = BASE_URL.format(page)
            print(f"\n🔹 Đang cào trang {page}: {page_url}")

            try:
                driver.get(page_url)
                WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
                time.sleep(1 + random.random() * 1.5)
            except Exception as e:
                print("❌ Không load được trang index:", page_url, "|", e)
                page += 1
                # small backoff
                time.sleep(2 + random.random() * 2)
                continue

            # Lấy container bài
            try:
                container = driver.find_element(By.CSS_SELECTOR, "div.p-lst-articles")
                anchors = container.find_elements(By.TAG_NAME, "a")
            except Exception as e:
                print("⚠️ Không tìm thấy div.p-lst-articles ở trang", page, "|", e)
                page += 1
                continue

            # Lọc link bài viết (chỉ lấy links liên quan tới /kinh-doanh/ và bỏ các link index/page)
            links = []
            for a in anchors:
                try:
                    href = a.get_attribute("href")
                    if not href:
                        continue
                    href = href.strip()
                    # chỉ lấy link bài chứa '/kinh-doanh/' và không chứa 'page=' query
                    if ("laodong.vn" in href) and ("/kinh-doanh/" in href) and ("page=" not in href):
                        links.append(href)
                except:
                    continue

            # loại trùng trong trang, giữ thứ tự
            seen_local = set()
            page_links = []
            for l in links:
                if l not in seen_local:
                    seen_local.add(l)
                    page_links.append(l)

            print(f"   → Tìm thấy {len(page_links)} link khả thi trên trang.")

            # Crawl từng bài trong trang
            for link in page_links:
                if current_count >= TARGET:
                    break

                if link in seen_urls:
                    print("      - Đã có sẵn, bỏ qua:", link)
                    continue

                try:
                    art = extract_article(driver, link)
                    if art and art["context"]:
                        writer.writerow(art)
                        csv_file.flush()
                        seen_urls.add(link)
                        current_count += 1
                        total_added += 1
                        print("      ✅ Thêm:", link, f"(tổng hiện tại: {current_count})")
                    else:
                        print("      ⚠️ Bỏ qua (không có nội dung):", link)
                except Exception as e:
                    print("      ⚠️ Lỗi khi xử lý:", link, "|", e)

                time.sleep(1 + random.random() * 1.5)

            # tăng trang, chờ 1-3s
            page += 1
            time.sleep(2 + random.random() * 2.5)

        if current_count >= TARGET:
            print(f"\n🎉 Đã thu đủ mục tiêu: {current_count} bài (thêm mới {total_added}).")
        else:
            print(f"\n⛔ Kết thúc do chạm MAX_PAGE={MAX_PAGE}. Tổng bài hiện tại: {current_count} (thêm mới {total_added}).")
    finally:
        csv_file.close()
        try:
            driver.quit()
        except:
            pass

if __name__ == "__main__":
    crawl()


🔸 Đã có sẵn 516 bài trong laodong_congnghe.csv

🔹 Đang cào trang 1: https://laodong.vn/kinh-doanh?page=1
   → Tìm thấy 8 link khả thi trên trang.
      ✅ Thêm: https://laodong.vn/kinh-doanh/doanh-nghiep-khat-san-choi-minh-bach-cho-tai-san-so-1623249.ldo (tổng hiện tại: 517)
      ✅ Thêm: https://laodong.vn/kinh-doanh/thach-thuc-trong-phat-trien-du-lich-mice-o-tphcm-1623312.ldo (tổng hiện tại: 518)
      ✅ Thêm: https://laodong.vn/kinh-doanh/cay-xoai-mo-huong-thoat-ngheo-cho-nong-dan-vung-bien-o-an-giang-1623201.ldo (tổng hiện tại: 519)
      ✅ Thêm: https://laodong.vn/kinh-doanh/gia-vang-duoc-du-bao-chuan-bi-but-pha-gioi-dau-tu-gom-hang-1623066.ldo (tổng hiện tại: 520)
      ✅ Thêm: https://laodong.vn/kinh-doanh/thue-thu-nhap-ca-nhan-thay-doi-lon-bieu-thue-5-bac-co-loi-hon-cho-nguoi-lao-dong-1623234.ldo (tổng hiện tại: 521)
      ✅ Thêm: https://laodong.vn/kinh-doanh/tphcm-dat-muc-tieu-thu-hon-2200-ti-dong-moi-ngay-trong-nam-2026-1623297.ldo (tổng hiện tại: 522)
      ✅ Thêm: https://l

In [1]:
import pandas as pd

# Đường dẫn file gốc
input_path = r"C:\Users\DELL\Desktop\The News\data\laodong_all_news.csv"
output_path = r"C:\Users\DELL\Desktop\The News\data\laodong_all_news_with_split.csv"

# Đọc dữ liệu
df = pd.read_csv(input_path)

# Khởi tạo toàn bộ là TRAIN
df["is_test"] = 0

# Lấy ngẫu nhiên 20% làm TEST
test_idx = df.sample(frac=0.15, random_state=42).index
df.loc[test_idx, "is_test"] = 1

# Ghi ra file
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("✅ ĐÃ GHI FILE:", output_path)
print(df["is_test"].value_counts())


✅ ĐÃ GHI FILE: C:\Users\DELL\Desktop\The News\data\laodong_all_news_with_split.csv
is_test
0    5100
1     900
Name: count, dtype: int64


In [5]:
import pandas as pd

# Đọc file CSV hiện có
file_path = "laodong_congnghe.csv"
df = pd.read_csv(file_path, encoding="utf-8-sig")

print(f"Số dòng ban đầu: {len(df)}")

# Loại bỏ các dòng trùng nhau theo cột 'url', giữ lại dòng đầu tiên
df = df.drop_duplicates(subset="url", keep="first").reset_index(drop=True)

print(f"Số dòng sau khi bỏ trùng: {len(df)}")

# Ghi đè lại file CSV
df.to_csv(file_path, index=False, encoding="utf-8-sig")

print(f"✅ Đã cập nhật file CSV, bỏ các bản trùng nhau theo URL.")


Số dòng ban đầu: 516
Số dòng sau khi bỏ trùng: 516
✅ Đã cập nhật file CSV, bỏ các bản trùng nhau theo URL.


In [7]:
import pandas as pd
import os

INPUT_CSV = "laodong_congnghe.csv"   # đổi tên file nếu cần
OUTPUT_CSV = INPUT_CSV  # ghi đè file cũ; đổi tên nếu muốn lưu ra file mới

# đọc CSV
df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")

# Chọn cột nội dung: ưu tiên 'content', nếu không có thì 'context', nếu không có thì tạo 'content'
if "content" in df.columns:
    content_col = "content"
elif "context" in df.columns:
    content_col = "context"
    # nếu muốn unify tên cột sang 'content', uncomment dòng dưới:
    # df = df.rename(columns={"context": "content"}); content_col = "content"
else:
    content_col = "content"
    df[content_col] = ""  # tạo cột content rỗng

# Đảm bảo cột 'title' tồn tại
if "title" not in df.columns:
    print("⚠️ Không tìm thấy cột 'title' trong file. Hãy kiểm tra lại.")
else:
    def ensure_title_in_content(row):
        title = str(row.get("title", "")).strip()
        content = "" if pd.isna(row.get(content_col)) else str(row.get(content_col, ""))
        # chuẩn hóa để so sánh: loại khoảng trắng đầu-cuối và chữ thường
        if not title:
            return content
        # so sánh không phân biệt hoa thường, bỏ khoảng trắng đầu-cuối
        if title.lower().strip() in content.lower().strip():
            # đã có title trong content -> giữ nguyên content
            return content
        # nếu content rỗng -> chỉ chép title
        if not content.strip():
            return title
        # ngược lại thêm title + 2 dòng trống + content hiện có
        return title + "\n\n" + content

    # áp dụng cho từng dòng
    df[content_col] = df.apply(ensure_title_in_content, axis=1)

    # (tùy chọn) nếu bạn muốn xoá cột title sau khi chuyển, bỏ comment dòng dưới:
    # df = df.drop(columns=["title"])

    # nếu bạn muốn chuẩn hóa tên cột là 'content' (nếu trước đó dùng 'context')
    if content_col == "context":
        df = df.rename(columns={"context": "content"})
        content_col = "content"

    # lưu lại (ghi đè)
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"✅ Đã cập nhật cột '{content_col}' và lưu vào: {OUTPUT_CSV}")
    print(f"Số dòng: {len(df)}")


✅ Đã cập nhật cột 'content' và lưu vào: laodong_congnghe.csv
Số dòng: 1500


In [3]:
import pickle

with open("C:\\Users\\DELL\\Desktop\\The News\\model\\label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

print(le.classes_)


['cong-nghe' 'kinh-doanh' 'the-gioi' 'the-thao']


c:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [8]:
import pandas as pd
from pathlib import Path

# ==== Thay đổi đường dẫn nếu cần ====
files_and_labels = {
    r"C:\Users\DELL\Desktop\The News\laodong_congnghe.csv": "congnghe",
    r"C:\Users\DELL\Desktop\The News\laodong_kinhdoanh.csv": "kinhdoanh",
    r"C:\Users\DELL\Desktop\The News\laodong_thegioi.csv": "thegioi",
    r"C:\Users\DELL\Desktop\The News\laodong_thethao.csv": "thethao",
}
OUT_FILE = r"C:\Users\DELL\Desktop\The News\laodong_all_combined.csv"
ENCODING = "utf-8-sig"
# =====================================

def prepare_df(path, label):
    p = Path(path)
    if not p.exists():
        print(f"⚠️ File không tồn tại: {path}")
        return pd.DataFrame()
    df = pd.read_csv(path, encoding=ENCODING)
    # Nếu dùng 'context' thay vì 'content', chuyển tên cột
    if "context" in df.columns and "content" not in df.columns:
        df = df.rename(columns={"context": "content"})
    # Đảm bảo có các cột cơ bản
    for c in ["url", "title", "content"]:
        if c not in df.columns:
            df[c] = ""
    # Fill NaN
    df["url"] = df["url"].fillna("").astype(str)
    df["title"] = df["title"].fillna("").astype(str)
    df["content"] = df["content"].fillna("").astype(str)
    # Thêm cột class (loại)
    df["class"] = label
    # Giữ thứ tự cột mong muốn (tuỳ bạn có thể để nhiều cột hơn)
    cols = ["url", "title", "content", "class"]
    # Nếu file có cột khác bạn muốn giữ, bỏ dòng tiếp theo
    df = df[cols]
    print(f"Đã đọc {len(df)} dòng từ {p.name} -> class = '{label}'")
    return df

def main():
    dfs = []
    for path, label in files_and_labels.items():
        df = prepare_df(path, label)
        if not df.empty:
            dfs.append(df)

    if not dfs:
        print("Không có file hợp lệ để ghép. Kết thúc.")
        return

    combined = pd.concat(dfs, ignore_index=True)
    print(f"\nTổng dòng sau khi ghép: {len(combined)}")

    # --- Nếu bạn muốn loại bỏ các url trùng, mở dòng dưới này ---
    # combined = combined.drop_duplicates(subset="url", keep="first").reset_index(drop=True)
    # print(f"Tổng dòng sau khi bỏ trùng theo url: {len(combined)}")

    combined.to_csv(OUT_FILE, index=False, encoding=ENCODING)
    print(f"\n✅ Đã ghi file hợp nhất: {OUT_FILE}")
    print("\nThống kê theo class:")
    print(combined["class"].value_counts())

if __name__ == "__main__":
    main()


Đã đọc 1500 dòng từ laodong_congnghe.csv -> class = 'congnghe'
Đã đọc 1500 dòng từ laodong_kinhdoanh.csv -> class = 'kinhdoanh'
Đã đọc 1500 dòng từ laodong_thegioi.csv -> class = 'thegioi'
Đã đọc 1500 dòng từ laodong_thethao.csv -> class = 'thethao'

Tổng dòng sau khi ghép: 6000

✅ Đã ghi file hợp nhất: C:\Users\DELL\Desktop\The News\laodong_all_combined.csv

Thống kê theo class:
class
congnghe     1500
kinhdoanh    1500
thegioi      1500
thethao      1500
Name: count, dtype: int64


In [6]:
cong = pd.read_csv("laodong_kinhdoanh.csv")
cong

,url,title,context
0,https://laodong.vn/kinh-doanh/vi-pham-ve-bao-c...,"Vi phạm về báo cáo tài chính và trái phiếu, ha...","Vi phạm về báo cáo tài chính và trái phiếu, ha..."
1,https://laodong.vn/tien-te-dau-tu/cap-nhat-gia...,Cập nhật giá vàng sáng 5.12: Vàng trong nước s...,Cập nhật giá vàng sáng 5.12: Vàng trong nước s...
2,https://laodong.vn/kinh-doanh/lai-suat-ngan-ha...,Lãi suất ngân hàng hôm nay 5.12: Tăng kịch trần,Lãi suất ngân hàng hôm nay 5.12: Tăng kịch trầ...
3,https://laodong.vn/kinh-doanh/co-phieu-mbb-tan...,"Cổ phiếu MBB tăng 7 phiên liên tiếp, thanh kho...","Cổ phiếu MBB tăng 7 phiên liên tiếp, thanh kho..."
4,https://laodong.vn/kinh-doanh/an-giang-day-man...,An Giang đẩy mạnh quảng bá sản phẩm tại Hội ch...,An Giang đẩy mạnh quảng bá sản phẩm tại Hội ch...
...,...,...,...
1658,https://laodong.vn/kinh-doanh/tphcm-chu-dong-n...,"TPHCM chủ động nguồn cung hàng hóa, không lo s...","TPHCM chủ động nguồn cung hàng hóa, không lo s..."
1659,https://laodong.vn/kinh-doanh/gia-xang-dau-hom...,"Giá xăng dầu hôm nay 27.10: Tăng mạnh, xăng dầ...","Giá xăng dầu hôm nay 27.10: Tăng mạnh, xăng dầ..."
1660,https://laodong.vn/tien-te-dau-tu/bien-dong-gi...,Biến động giá bạc 27.10: Tiếp đà giảm sâu,Biến động giá bạc 27.10: Tiếp đà giảm sâu\n\nG...
1661,https://laodong.vn/tien-te-dau-tu/gia-vang-hom...,"Giá vàng hôm nay 27.10: Trong nước giằng co, t...","Giá vàng hôm nay 27.10: Trong nước giằng co, t..."


In [9]:
kinhd = pd.read_csv("laodong_thegioi.csv")
kinhd

,url,title,context
0,https://laodong.vn/the-gioi/nguoc-dong-eu-ital...,"Ngược dòng EU, Italy bước một chân khỏi cam kế...","Ngược dòng EU, Italy bước một chân khỏi cam kế..."
1,https://laodong.vn/the-gioi/ong-putin-khang-di...,Ông Putin khẳng định Nga không có ý định quay ...,Ông Putin khẳng định Nga không có ý định quay ...
2,https://laodong.vn/the-gioi/tien-than-bao-so-1...,Tiền thân bão số 16 dự kiến gây mưa to 3 ngày ...,Tiền thân bão số 16 dự kiến gây mưa to 3 ngày ...
3,https://laodong.vn/the-gioi/tong-thong-putin-n...,Tổng thống Putin: Nga sẽ kiểm soát toàn bộ vùn...,Tổng thống Putin: Nga sẽ kiểm soát toàn bộ vùn...
4,https://laodong.vn/podcast-tin-tuc/the-gioi-24...,"Thế giới 24h: Tiền thân bão số 16 mạnh lên, dự...","Thế giới 24h: Tiền thân bão số 16 mạnh lên, dự..."
...,...,...,...
1495,https://laodong.vn/the-gioi/singapore-lap-ky-l...,"Singapore lập kỷ lục dân số 6,1 triệu người","Singapore lập kỷ lục dân số 6,1 triệu người\n\..."
1496,https://laodong.vn/the-gioi/lien-minh-chau-au-...,Liên minh châu Âu chính thức tái áp đặt trừng ...,Liên minh châu Âu chính thức tái áp đặt trừng ...
1497,https://laodong.vn/the-gioi/phat-hien-thi-the-...,Phát hiện thi thể người trong càng đáp máy bay...,Phát hiện thi thể người trong càng đáp máy bay...
1498,https://laodong.vn/the-gioi/du-bao-mua-lon-gio...,"Dự báo mưa lớn, gió dữ dội vì siêu bão Gabrielle","Dự báo mưa lớn, gió dữ dội vì siêu bão Gabriel..."


In [17]:
thethao = pd.read_csv("laodong_thethao.csv")

In [10]:
kinhd.isnull().sum()

url        0
title      0
context    0
dtype: int64

In [16]:
dups = cong["url"][cong["url"].duplicated()].unique()
dups.shape


(328,)

In [18]:
dups = thethao["url"][thethao["url"].duplicated()].unique()
dups.shape


(332,)